In [7]:
import os
import numpy as np
import soundfile as sf
import pandas as pd
import librosa

# 1. Create Synthetic Catalog
os.makedirs("catalog", exist_ok=True)

moods = ['happy', 'sad', 'energetic', 'calm']
catalog_data = []

np.random.seed(42)
for i in range(20):
    mood = np.random.choice(moods)
    sr = 22050
    duration = 5 # seconds
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    
    # Generate basic tones based on mood to simulate different audio profiles
    if mood == 'happy':
        bpm = np.random.randint(100, 140)
        y = np.sin(2 * np.pi * 440 * t) * np.exp(-t)
    elif mood == 'sad':
        bpm = np.random.randint(60, 90)
        y = np.sin(2 * np.pi * 220 * t) * np.exp(-t*0.5)
    elif mood == 'energetic':
        bpm = np.random.randint(130, 180)
        y = np.sin(2 * np.pi * 880 * t) * np.random.rand(len(t))
    else: # calm
        bpm = np.random.randint(50, 80)
        y = np.sin(2 * np.pi * 330 * t)
        
    file_path = f"catalog/track_{i}_{mood}.wav"
    sf.write(file_path, y, sr)
    catalog_data.append({'track_id': f'track_{i}', 'file_path': file_path, 'true_mood': mood, 'true_bpm': bpm})

df_catalog = pd.DataFrame(catalog_data)
df_catalog.to_csv("catalog_metadata.csv", index=False)
print("Generated synthetic catalog of 20 tracks.")

Generated synthetic catalog of 20 tracks.


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# 2. Feature Extraction
def extract_features(file_path):
    try:
        y, sr = librosa.load(file_path, duration=30)
        # Extract MFCCs
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfcc_mean = np.mean(mfccs.T, axis=0)
        
        # Extract Tempo
        onset_env = librosa.onset.onset_strength(y=y, sr=sr)
        tempo, _ = librosa.beat.beat_track(onset_envelope=onset_env, sr=sr)
        if isinstance(tempo, np.ndarray):
             tempo = tempo[0]
             
        # Energy
        rmse = np.mean(librosa.feature.rms(y=y))
        
        return np.concatenate((mfcc_mean, [tempo, rmse]))
    except Exception as e:
        return None

# Extract features for catalog
print("Extracting features for catalog...")
features_list = []
valid_indices = []
for idx, row in df_catalog.iterrows():
    feats = extract_features(row['file_path'])
    if feats is not None:
        features_list.append(feats)
        valid_indices.append(idx)

X = np.array(features_list)
y_labels = df_catalog.iloc[valid_indices]['true_mood'].values

# 3. Train Classifier
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

clf = LogisticRegression(max_iter=500, random_state=42)
clf.fit(X_scaled, y_labels)

# Save precomputed features to catalog dataframe
df_catalog['features'] = None
df_catalog['extracted_bpm'] = 0.0
for i, idx in enumerate(valid_indices):
    df_catalog.at[idx, 'features'] = X_scaled[i]
    df_catalog.at[idx, 'extracted_bpm'] = X[i][-2] # Tempo is second to last feature

print("Classifier trained and features precomputed!")

Extracting features for catalog...
Classifier trained and features precomputed!


In [9]:
import sqlite3
import datetime

# 4. SQLite Persistence Setup
DB_PATH = 'playlist_queries.db'

def init_db():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS queries (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp TEXT,
            detected_mood TEXT,
            detected_bpm REAL,
            recommended_tracks TEXT
        )
    ''')
    conn.commit()
    conn.close()

def log_query(mood, bpm, tracks):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    track_ids = ",".join(tracks)
    cursor.execute('INSERT INTO queries (timestamp, detected_mood, detected_bpm, recommended_tracks) VALUES (?, ?, ?, ?)',
                   (datetime.datetime.now().isoformat(), mood, float(bpm), track_ids))
    conn.commit()
    conn.close()

init_db()
print("Database initialized.")

Database initialized.


In [ ]:
import gradio as gr
import matplotlib.pyplot as plt
import io
from PIL import Image

# 5. Matching Logic and UI
def plot_spectrogram(y, sr):
    fig, ax = plt.subplots(figsize=(6, 3))
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    img = librosa.display.specshow(D, y_axis='log', x_axis='time', sr=sr, ax=ax)
    fig.colorbar(img, ax=ax, format="%+2.0f dB")
    ax.set_title('Power Spectrogram')
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf)

def process_audio(audio_path, mood_weight):
    if audio_path is None:
        return "Please upload an audio file.", None, None, [], None
    
    y, sr = librosa.load(audio_path, duration=30)
    spec_img = plot_spectrogram(y, sr)
    
    feats = extract_features(audio_path)
    if feats is None:
        return "Error processing audio.", None, None, [], spec_img
        
    feats_scaled = scaler.transform([feats])
    
    # Mood Prediction
    mood_probs = clf.predict_proba(feats_scaled)[0]
    max_prob_idx = np.argmax(mood_probs)
    pred_mood = clf.classes_[max_prob_idx]
    confidence = mood_probs[max_prob_idx]
    
    if confidence < 0.4:
        mood_display = f"Not Confident (Fallback: {pred_mood}, {confidence:.2f})"
    else:
        mood_display = f"{pred_mood} ({confidence:.2f})"
        
    bpm = feats[-2]
    
    # Matching
    results = []
    for idx, row in df_catalog.iterrows():
        if row['features'] is None: continue
        
        # Tempo check (+/- 8%)
        cat_bpm = row['extracted_bpm']
        tempo_diff_ratio = abs(bpm - cat_bpm) / max(bpm, 1)
        
        # Feature distance
        feat_dist = np.linalg.norm(feats_scaled[0] - row['features'])
        
        # Mood match score (1 if match, 0 if not)
        mood_score = 1.0 if row['true_mood'] == pred_mood else 0.0
        
        # Combined Score
        tempo_score = max(0, 1 - tempo_diff_ratio*5) # Penalize tempo differences
        
        # Incorporate slider weight (mood_weight: 0 means all tempo, 1 means all mood)
        combined_score = (mood_weight * mood_score) + ((1 - mood_weight) * tempo_score) - (0.1 * feat_dist)
        
        results.append((row['track_id'], row['file_path'], combined_score, cat_bpm, row['true_mood']))
        
    # Sort and pick top 5
    results.sort(key=lambda x: x[2], reverse=True)
    top_5 = results[:5]
    
    # Log
    log_query(pred_mood, bpm, [t[0] for t in top_5])
    
    output_text = f"**Detected BPM:** {bpm:.1f}\n**Predicted Mood:** {mood_display}"
    audio_outputs = [t[1] for t in top_5]
    while len(audio_outputs) < 5:
        audio_outputs.append(None)
        
    return output_text, audio_outputs[0], audio_outputs[1], audio_outputs[2], audio_outputs[3], audio_outputs[4], spec_img

with gr.Blocks(title="Smart Playlist Matcher") as demo:
    gr.Markdown("# Smart Playlist Matcher \n Upload a 10-60s clip to find similar tracks based on mood and tempo!")
    
    with gr.Row():
        with gr.Column():
            audio_in = gr.Audio(type="filepath", label="Upload Audio Clip")
            mood_weight_slider = gr.Slider(0.0, 1.0, value=0.7, label="Weight: Tempo (0) <---> Mood (1)")
            btn = gr.Button("Find Matches")
        with gr.Column():
            info_out = gr.Markdown(label="Analysis Results")
            spec_out = gr.Image(label="Audio Spectrogram")
            
    gr.Markdown("### Top 5 Recommended Tracks")
    track1 = gr.Audio(label="Match 1")
    track2 = gr.Audio(label="Match 2")
    track3 = gr.Audio(label="Match 3")
    track4 = gr.Audio(label="Match 4")
    track5 = gr.Audio(label="Match 5")
    
    btn.click(fn=process_audio, 
              inputs=[audio_in, mood_weight_slider], 
              outputs=[info_out, track1, track2, track3, track4, track5, spec_out])

demo.launch(debug=True, share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://cba248790b8ee4bbc9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "c:\Users\brian\anaconda3\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "c:\Users\brian\anaconda3\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "c:\Users\brian\anaconda3\Lib\site-packages\gradio\blocks.py", line 2169, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\brian\anaconda3\Lib\site-packages\gradio\blocks.py", line 1877, in postprocess_data
    self.validate_outputs(block_fn, predictions)  # type: ignore
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\brian\anaco